# DSA 4060 – Week 3 Lab: Building a Simple Content-Based Recommender

**Lab objective:** Build a content-based movie recommender using structured genre features: movie features → user preferences → recommendation scores → Top-N recommendations.

## 2. Import Libraries

In [1]:
import pandas as pd

## 3. Create / Load Dataset

In [2]:
movies = pd.DataFrame({
    "movie_id": [1,2,3,4,5,6,7,8,9,10],
    "title": [
        "Galaxy Mission","Love in Nairobi","Final Battle","Family Weekend",
        "Future World","Broken Hearts","City Detectives","Space Warriors",
        "Wedding Weekend","Hidden Truth"
    ],
    "Action":  [1,0,1,0,1,0,1,1,0,0],
    "Comedy":  [0,0,0,1,0,0,0,0,1,0],
    "Drama":   [0,1,0,0,0,1,1,0,0,1],
    "Romance": [0,1,0,0,0,1,0,0,1,0],
    "SciFi":   [1,0,0,0,1,0,0,1,0,0]
})
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


In [3]:
movies.to_csv("movies.csv", index=False)

## 4. Explore the Dataset

In [4]:
movies.head()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


In [5]:
movies.shape

(10, 7)

In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 692.0+ bytes


**Step 5 questions**
1. How many movies are in the dataset?
2. How many genre features exist?
3. What does one row represent?

**Answer:**


1. The dataset has **10 movies** (11 after I add my own movie in Step 24).
2. There are **5 genre features**: Action, Comedy, Drama, Romance and SciFi.
3. **One row represents one movie**: its ID, its title and five binary (1/0) flags showing which genres it has.

## 5. Identify Structured Features

In [7]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


## 6. Create Item Profiles

In [8]:
movies.loc[movies["title"] == "Galaxy Mission"]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


**Step 7:** What do the 1s and 0s in this movie's profile mean?

**Answer:**


Galaxy Mission has 1s for Action and SciFi, so it belongs to both genres. The 0s for Comedy, Drama and Romance mean it has none of those genres, giving the item profile [1, 0, 0, 0, 1].

## 7. Create User Ratings

In [9]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating":   [5, 2, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


In [10]:
user_movies = ratings.merge(movies, on="movie_id")
user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


In [11]:
user_movies[["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


**Step 10 observation:** What kinds of movies does User A appear to prefer?

**Answer:**


User A gave 5 stars to Galaxy Mission and Future World, and 4 to Final Battle. All three are Action movies, and two are also SciFi. The only low rating (2) went to Love in Nairobi, a Drama/Romance film. So User A prefers **Action and Sci-Fi** and seems less interested in Drama and Romance.

## 8. Build the User Profile

In [12]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]
rated_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,1,0,0,0,1


In [13]:
user_ratings

0    5
1    2
2    4
3    5
Name: rating, dtype: int64

In [14]:
weighted_features = rated_features.mul(user_ratings, axis=0)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


In [15]:
user_profile = weighted_features.sum()
user_profile

Action     14
Comedy      0
Drama       2
Romance     2
SciFi      10
dtype: int64

In [16]:
user_profile.sort_values(ascending=False)

Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

**Step 14:** Which two genres appear strongest? Explain using the profile scores.

**Answer:**


The two strongest genres are **Action (14)** and **SciFi (10)**. Action appears in three liked movies (5 + 4 + 5 = 14) and SciFi in two 5-star movies (5 + 5 = 10). Drama and Romance only score 2 each, and Comedy scores 0.

In [17]:
user_profile_normalized = user_profile / user_profile.sum()
user_profile_normalized

Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64

## 9. Calculate Recommendation Scores

In [18]:
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)
movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


In [19]:
ranked_movies = movies.sort_values("recommendation_score", ascending=False)
ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


## 10. Remove Watched Movies

In [20]:
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[~ranked_movies["movie_id"].isin(watched_movies)]
recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


## 11. Generate Top-N Recommendations

In [21]:
top_3 = recommendations[["title", "recommendation_score"]].head(3)
top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


**Step 20:** One sentence per Top 3 movie explaining why it was recommended, referring to the user's strongest preferences.

**Answer:**


- **Space Warriors (0.857):** It is Action and SciFi, User A's two strongest genres, which together make up about 86% of the normalized profile (0.50 + 0.357).
- **City Detectives (0.571):** It is Action, the strongest preference (0.50), plus Drama, which adds a small 0.071.
- **Broken Hearts (0.143):** It is Drama and Romance, which are weak preferences (0.071 each). It only appears because few unwatched Action/SciFi movies are left, so this is a weak recommendation.

In [22]:
def recommend_movies(movies, ratings, features, n=3):
    user_movies = ratings.merge(movies, on="movie_id")
    rated_features = user_movies[features]
    weighted_features = rated_features.mul(user_movies["rating"], axis=0)
    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()
    scores = movies[features].mul(user_profile).sum(axis=1)
    results = movies.copy()
    results["recommendation_score"] = scores
    watched = ratings["movie_id"]
    results = results[~results["movie_id"].isin(watched)]
    return results.sort_values("recommendation_score", ascending=False).head(n)

In [23]:
recommend_movies(movies, ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


## 12. Test a Second User

In [24]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating":   [5, 4, 5, 4]
})
recommend_movies(movies, user_b_ratings, features)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


**Step 22:** Compare User A and User B. Same recommendations? Why or why not?

**Answer:**


No, they received different recommendations. User A's profile is dominated by Action (0.50) and SciFi (0.357), so Space Warriors, City Detectives and Broken Hearts came out on top. User B rated Romance, Drama and Comedy movies highly (Romance 14, Drama 10, Comedy 8, Action and SciFi 0). Their top picks were Hidden Truth and City Detectives (tied at 0.3125, both through Drama), then Final Battle at 0. The recommender scores movies by how well their genres match each user's own profile, so different histories give different results. Final Battle scoring 0 shows that B's list runs out of relevant movies quickly.

### Thinking Exercise (Step 23): User C

In [25]:
user_c_ratings = pd.DataFrame({"movie_id": [1], "rating": [5]})
rated = user_c_ratings.merge(movies, on="movie_id")
profile_c = rated[features].mul(rated["rating"], axis=0).sum()
profile_c = profile_c / profile_c.sum()
print(profile_c)
recommend_movies(movies, user_c_ratings, features)

Action     0.5
Comedy     0.0
Drama      0.0
Romance    0.0
SciFi      0.5
dtype: float64


,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
4,5,Future World,1,0,0,0,1,1.0
7,8,Space Warriors,1,0,0,0,1,1.0
6,7,City Detectives,1,0,1,0,0,0.5


**Step 23 questions**
4. What will the model learn about User C?
5. Which genres will dominate the profile?
6. What could go wrong with the recommendations?
7. Is one rating enough to understand a user's preferences?

**Answer:**


4. The model learns only that User C likes the genres of Galaxy Mission, Action and SciFi.
5. **Action (0.5) and SciFi (0.5)** dominate the profile, and all other genres are 0.
6. The recommendations are based on a single movie, so they will be very narrow and too similar to it. Future World and Space Warriors tie at 1.0, so there is no meaningful ranking between them. A 5-star rating does not say *why* the user liked the movie (genre, actor, story), so the model may be wrong about the real reason.
7. No. One rating is a cold-start situation. There is not enough evidence to separate real preferences from coincidence, so more and more varied ratings are needed.

## 13. Add Your Unique Movie

> Edit the title and feature values below so the movie is your own before running.

In [26]:
new_movie = {
    "movie_id": 11,
    "title": "Silent Orbit",   # a thoughtful sci-fi drama
    "Action": 0,
    "Comedy": 0,
    "Drama": 1,
    "Romance": 0,
    "SciFi": 1
}

movies = pd.concat([movies, pd.DataFrame([new_movie])], ignore_index=True)
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
0,1,Galaxy Mission,1,0,0,0,1,0.857143
1,2,Love in Nairobi,0,0,1,1,0,0.142857
2,3,Final Battle,1,0,0,0,0,0.500000
3,4,Family Weekend,0,1,0,0,0,0.000000
4,5,Future World,1,0,0,0,1,0.857143
5,6,Broken Hearts,0,0,1,1,0,0.142857
6,7,City Detectives,1,0,1,0,0,0.571429
7,8,Space Warriors,1,0,0,0,1,0.857143
8,9,Wedding Weekend,0,1,0,1,0,0.071429
9,10,Hidden Truth,0,0,1,0,0,0.071429


In [27]:
movies.drop(columns="recommendation_score", errors="ignore").to_csv("movies.csv", index=False)

## 14. Create Your Own User (Hans)

> Hans enjoys dramas and action. He rated four movies: Future World (4), City Detectives (5), Hidden Truth (5) and Family Weekend (1).

In [28]:
hans = pd.DataFrame({
    "movie_id": [5, 7, 10, 4],
    "rating":   [4, 5, 5, 1]
})
hans_recs = recommend_movies(movies, hans, features, n=5)
hans_recs[["title", "recommendation_score"]]

,title,recommendation_score
10,Silent Orbit,0.583333
7,Space Warriors,0.541667
0,Galaxy Mission,0.541667
5,Broken Hearts,0.416667
1,Love in Nairobi,0.416667


In [29]:
# Hans's normalized profile (for the interpretation questions)
hm = hans.merge(movies, on="movie_id")
hans_profile = hm[features].mul(hm["rating"], axis=0).sum()
(hans_profile / hans_profile.sum()).sort_values(ascending=False)

Drama      0.416667
Action     0.375000
SciFi      0.166667
Comedy     0.041667
Romance    0.000000
dtype: float64

## 15. Interpretation

**Step 26 questions**
8. What are your user's two strongest preferences?
9. Which three movies received the highest recommendation scores?
10. Why did the top movie rank first?
11. Did any recommendation surprise you?
12. What important information is the model ignoring?

**Answer:**


8. Hans's two strongest preferences are **Drama (0.417)** and **Action (0.375)**, followed by SciFi (0.167).
9. The three highest scores are **Silent Orbit (0.583)**, then **Space Warriors** and **Galaxy Mission**, tied at 0.542.
10. Silent Orbit ranked first because it is Drama and SciFi. It matches Hans's strongest genre (Drama, 0.417) and also his SciFi interest (0.167), which adds up to 0.583.
11. Yes, two things. First, Space Warriors and Galaxy Mission ranked above Love in Nairobi and Broken Hearts, although Drama is Hans's top genre, because Action is a close second and these two match both Action and SciFi. Second, Hans gave Family Weekend only 1 star, but it still added a small positive weight to Comedy (0.042), because the model treats every rating as positive evidence.
12. The model ignores negative feedback (a low rating should count against a genre), actors, directors, release year, language, plot descriptions, how popular a movie is, and when or how the user watched it.

## 16. Limitations

Identify at least three limitations of this recommender.

**Answer:**


1. **Only five genre features are used.** Two movies with the same genres look identical, even if they are very different in quality or style (for example Galaxy Mission, Future World and Space Warriors all tie).
2. **Rich metadata is ignored.** Actors, directors, release year, language and plot descriptions are not used, so the model cannot learn that a user likes a certain actor or era.
3. **Recommendations are too similar to past likes.** The model keeps pushing the same genres, which creates a filter bubble and little discovery.
4. **Cold start for new users.** A user with one or no ratings gets a very narrow or empty profile (see User C).
5. **Low ratings still add positive weight.** A 1-star rating still increases the score of that movie's genres, because ratings are only used as positive weights.
6. **Poor metadata gives poor recommendations.** If genres are wrong or missing, the profiles and scores are wrong too.

## 17. Conclusion

In this lab I built a simple content-based recommender using five binary genre features. Ratings were used to weight each genre and build a normalized user profile. Every movie was scored against that profile, movies the user had already watched were removed, and the highest-scoring ones were returned as Top-N recommendations. User A (Action/SciFi), User B (Romance/Drama/Comedy) and my own user Hans (Drama/Action) each got different recommendations, which shows the model personalises to each profile. The approach is simple and easy to explain, since every score can be traced back to a genre. Its main weaknesses are the small feature set, the lack of any negative signal, the cold start for new users and the risk of recommending only more of the same. Week 4 techniques such as TF-IDF and cosine similarity can handle richer descriptions.

### Final Question

If two users have watched exactly the same movies but gave them different ratings, should this recommender produce the same recommendations for both?

In [30]:
# Same movies as User A (1, 2, 3, 5) but very different ratings
user_d_ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating":   [2, 5, 2, 2]
})
print("User A:")
display(recommend_movies(movies, ratings, features)[["title", "recommendation_score"]])
print("User D (same movies, different ratings):")
display(recommend_movies(movies, user_d_ratings, features)[["title", "recommendation_score"]])

User A:


,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
10,Silent Orbit,0.428571


User D (same movies, different ratings):


,title,recommendation_score
6,City Detectives,0.55
5,Broken Hearts,0.50
7,Space Warriors,0.50


**Final answer:**


No. The two users would have the same movies removed as "already watched", but the ratings act as weights when the user profile is built, so different ratings give a different profile and different scores. In my test, User A and User D rated the same four movies (Galaxy Mission, Love in Nairobi, Final Battle, Future World). User A gave 5, 2, 4, 5 and got an Action 0.50 / SciFi 0.357 profile, with Space Warriors ranked first (0.857). User D gave 2, 5, 2, 2 and got a much flatter profile (Action 0.30, Drama 0.25, Romance 0.25, SciFi 0.20), with City Detectives ranked first (0.55). The scores and the ranking changed even though the watched list was identical. The only case where the results would match is if one user's ratings were the same as the other's multiplied by a constant (for example 2, 4, 4, 2 and 1, 2, 2, 1), because normalization removes the overall scale.